# 04 Semantic vector search

**Question:** How do we match descriptions by meaning rather than exact words? MiniLM maps queries and book blurbs into the same 384-dimensional space. LangChain supplies document and vector-store interfaces; Chroma stores the actual vectors.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "book_recommender").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from book_recommender.config import RAW, CLEAN, CLASSIFIED, ENRICHED, REPORTS


In [2]:
from book_recommender.vector_search import LocalEmbeddings
embedding = LocalEmbeddings()
vectors = embedding.embed_documents(["A voyage to distant planets", "An adventure in outer space", "Cooking soup at home"])
import numpy as np
np.asarray(vectors).shape

C:\Users\atitk\OneDrive\Documents\ChatGPT\LLM_Semantic_Book_Recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(3, 384)

## Cosine similarity
For normalized vectors, the dot product equals cosine similarity. A score is geometric similarity, not the chance that a reader will like the book. MiniLM truncates long descriptions at its configured sequence length; check it below.

In [3]:
print("Token limit:", embedding.model.max_seq_length)
np.asarray(vectors) @ np.asarray(vectors).T

Token limit: 256


array([[1.00000006, 0.55640198, 0.00909684],
       [0.55640198, 1.00000008, 0.0325064 ],
       [0.00909684, 0.0325064 , 1.00000015]])

## Persistent storage
Run `python -m scripts.build_pipeline --stage index` once after enrichment. One document per book carries its identifier and category in metadata. This avoids parsing IDs out of text and keeps repeated runs idempotent.

In [4]:
from book_recommender.vector_search import BookRecommender
engine = BookRecommender(embeddings=embedding)
engine.manifest

{'collection': 'books-3459e8cac7a047a2-e9e2c881',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'data_sha256': '3459e8cac7a047a296ffb422b99c6c965461e9a6a130e6dd7eac519e7ad49a98',
 'book_count': 5729,
 'distance': 'cosine'}

In [5]:
results = engine.search("A journey through space and the search for other life", category="All", top_k=5)
results[["title", "predicted_category", "similarity", "rank_score"]]

,title,predicted_category,similarity,rank_score
0,Wasteland of Flint,Fiction,0.581312,0.790656
1,A Brief History of Time,Non-fiction,0.533518,0.766759
2,Existential Meditation,Non-fiction,0.527680,0.763840
3,The Mantle of Kendis-Dai,Fiction,0.526685,0.763342
4,Unto Leviathan,Fiction,0.525284,0.762642


## Filter before retrieval and rerank after
The category is a Chroma metadata filter. Mood blends a normalized similarity with an emotion probability within a semantic shortlist. Increasing mood influence can lower topical relevance, so show both controls and scores.

In [6]:
engine.search("A family rebuilding their lives", tone="Happy", weight=0.4, top_k=5)[["title", "similarity", "joy", "rank_score"]]

,title,similarity,joy,rank_score
0,The Blackwater Lightship,0.480355,0.879718,0.795993
1,Four Mothers,0.387553,0.611136,0.660720
2,The Horizontal World,0.458425,0.421354,0.606069
3,Novels,0.403587,0.413179,0.586348
4,A Feather on the Breath of God,0.424630,0.320351,0.555529


## Extension exercise
Try chunked embeddings for long blurbs, deduplicating hits by ISBN. Compare against the single-vector baseline using human judgments before deciding it is better.